# 34회차 Part E — 긍정·부정 키워드 비교

> **실습 1~4단계가 완료된 상태에서 실행하세요.**  
> `rev`, `counter_to_df`, `peek` 등이 이미 정의되어 있어야 합니다.

In [ ]:
# ── 사전 준비: 앞 단계 코드가 없으면 여기서 전체 파이프라인 재실행 ──
# (이미 실행했다면 이 셀은 건너뜁니다)

!pip install kiwipiepy -q

import re
import pandas as pd
from collections import Counter
from kiwipiepy import Kiwi

kiwi = Kiwi()

def clean_text(text):
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def tokenize_ko(text):
    return [t.form for t in kiwi.tokenize(text)
            if t.tag in ("NNG", "NNP", "VV", "VA")]

def counter_to_df(counter_obj, col="word"):
    return pd.DataFrame(counter_obj.most_common(), columns=[col, "count"])

def peek(df, n=5, title=""):
    if title:
        print(f"── {title} ──")
    print(df.head(n).to_string(index=False))

rev = pd.read_csv("/content/fintech_app_reviews.csv")
rev["clean"] = rev["review_text"].map(clean_text)
rev["tokens"] = rev["clean"].map(tokenize_ko)

stopwords = {"있", "하", "되", "수", "것", "거", "보", "같", "늦", "오"}
rev["tokens"] = rev["tokens"].map(lambda toks: [w for w in toks if w not in stopwords])

finance_terms = {
    "이체": "송금", "보내기": "송금",
    "본인확인": "인증", "신분확인": "인증",
    "에러": "오류", "실패": "오류",
}
rev["tokens"] = rev["tokens"].map(lambda toks: [finance_terms.get(w, w) for w in toks])

print("파이프라인 완료 — 전처리 샘플:")
peek(rev[["review_text", "tokens"]].head(3))

---
## E-1. 긍정·부정 상위 키워드 표

워크시트 E-1 표를 채울 데이터를 출력합니다.

In [ ]:
TOP_N = 10  # 워크시트 E-1은 5순위까지만 필요하지만 여유 있게 10개 출력

neg_tokens = Counter(
    w for toks in rev[rev["sentiment"] == "negative"]["tokens"] for w in toks
)
pos_tokens = Counter(
    w for toks in rev[rev["sentiment"] == "positive"]["tokens"] for w in toks
)

neg_top = neg_tokens.most_common(TOP_N)
pos_top = pos_tokens.most_common(TOP_N)

compare = pd.DataFrame({
    "순위":      list(range(1, TOP_N + 1)),
    "부정_키워드": [w for w, _ in neg_top],
    "부정_빈도":  [c for _, c in neg_top],
    "긍정_키워드": [w for w, _ in pos_top],
    "긍정_빈도":  [c for _, c in pos_top],
})

peek(compare, n=TOP_N, title="E-1 긍·부정 키워드 상위 비교")

---
## E-2. 해석 메모 — 원문 맥락 확인

특정 키워드가 포함된 원문을 뽑아 **긍/부정 맥락을 비교**합니다.  
`TARGET_WORD` 값을 바꿔가며 여러 단어를 확인하세요.

In [ ]:
TARGET_WORD = "송금"   # ← 확인하고 싶은 단어로 바꾸세요

mask = rev["tokens"].map(lambda toks: TARGET_WORD in toks)
context_df = rev[mask][["sentiment", "review_text"]].copy()
context_df.index = range(1, len(context_df) + 1)

print(f"['{TARGET_WORD}' 포함 리뷰 — 감성별 원문]")
for sent in ["negative", "neutral", "positive"]:
    sub = context_df[context_df["sentiment"] == sent]
    print(f"\n▶ {sent} ({len(sub)}건)")
    for _, row in sub.head(5).iterrows():
        print(f"  • {row['review_text']}")

---
## E-2 (속). 긍·부정 공통 출현 단어 확인

양쪽 모두에 나오는 단어가 있다면 **왜 그런지** 맥락으로 설명해야 합니다.

In [ ]:
neg_set = {w for w, _ in neg_tokens.most_common(15)}
pos_set = {w for w, _ in pos_tokens.most_common(15)}
common  = neg_set & pos_set

print("긍·부정 상위 15개 중 공통 단어:", sorted(common))

# 공통 단어 각각의 긍·부정 빈도 비교
if common:
    rows = []
    for w in sorted(common):
        rows.append({
            "단어":    w,
            "부정_빈도": neg_tokens[w],
            "긍정_빈도": pos_tokens[w],
            "우세":    "부정" if neg_tokens[w] > pos_tokens[w] else "긍정",
        })
    peek(pd.DataFrame(rows), n=20, title="공통 단어 긍·부정 빈도 비교")

---
## E-3. 단어 하나를 업무 액션으로 번역하기

부정 상위 키워드 중 하나를 골라 원인과 액션을 도출합니다.

In [ ]:
# 부정 상위 5개에 대한 앱별 빈도 → 어느 앱이 가장 문제인지 확인
TOP_NEG_WORDS = [w for w, _ in neg_tokens.most_common(5)]

neg_rev = rev[rev["sentiment"] == "negative"].copy()

rows = []
for word in TOP_NEG_WORDS:
    for app in rev["app_name"].unique():
        cnt = sum(
            word in toks
            for toks in neg_rev[neg_rev["app_name"] == app]["tokens"]
        )
        rows.append({"키워드": word, "앱": app, "부정_빈도": cnt})

action_df = pd.DataFrame(rows)
pivot = action_df.pivot_table(
    index="키워드", columns="앱", values="부정_빈도", aggfunc="sum", fill_value=0
)

peek(pivot.reset_index(), n=10, title="E-3 부정 키워드 × 앱별 빈도 (업무 액션 도출용)")

In [ ]:
# 최종 산출물 저장 (워크시트 E 완성 후 사용)
compare.to_csv("34_partE_keyword_compare.csv", index=False, encoding="utf-8-sig")
pivot.reset_index().to_csv("34_partE_action_pivot.csv", index=False, encoding="utf-8-sig")
print("저장 완료: 34_partE_keyword_compare.csv / 34_partE_action_pivot.csv")